# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshaamirmalik-sketch/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
# Load the March 2026 warehouse file
# Decision point: 2026-03-30

from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con = duckdb.connect()

df = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ga4_sessions,
    gsc_avg_position,
    scroll_events
FROM read_parquet(?)
WHERE report_date = '2026-03-30'
""", [file_path]).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 331230
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'gsc_avg_position', 'scroll_events']


In [9]:
# ML-07 — Baseline Action Score
# Decision point: 2026-03-30
#
# Rule:
# Prioritize pages that have:
#   1. meaningful search impressions,
#   2. a valid average position within the top 20,
#   3. CTR below 0.5%.
#
# This is a transparent baseline for review prioritization.
# It is not a prediction of future performance or refresh impact.

import numpy as np
import pandas as pd

baseline = df.copy()

# Keep only rows with enough observed search opportunity
# and a valid search position.
baseline = baseline[
    (baseline["gsc_impressions"] >= 20) &
    (baseline["gsc_avg_position"] > 0) &
    (baseline["gsc_avg_position"] <= 20)
].copy()

# CTR is expressed as a percentage.
baseline["ctr"] = (
    baseline["gsc_clicks"] / baseline["gsc_impressions"]
) * 100

# Low-CTR visible-page rule.
baseline["is_candidate"] = baseline["ctr"] < 0.5

baseline = baseline[baseline["is_candidate"]].copy()

# Score combines search opportunity and visibility.
# Higher impressions = more observed opportunity.
# Better position = stronger visibility.
baseline["volume_score"] = (
    np.log1p(baseline["gsc_impressions"]) /
    np.log1p(baseline["gsc_impressions"]).max()
)

baseline["position_score"] = (
    (20 - baseline["gsc_avg_position"]) / 20
).clip(lower=0)

baseline["score"] = (
    0.6 * baseline["volume_score"] +
    0.4 * baseline["position_score"]
) * 100

# One reason code and one action label.
baseline["reason_code"] = "LOW_CTR_VISIBLE"
baseline["action"] = "REVIEW_CTR"

# Highest-priority candidates first.
baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

print("Candidate rows:", len(baseline))
print("\nRule:")
print("impressions >= 20 AND position <= 20 AND CTR < 0.5%")
print("\nReason code:", baseline["reason_code"].iloc[0] if len(baseline) else "No candidates")
print("Action:", baseline["action"].iloc[0] if len(baseline) else "No candidates")

baseline.head(10)

Candidate rows: 38250

Rule:
impressions >= 20 AND position <= 20 AND CTR < 0.5%

Reason code: LOW_CTR_VISIBLE
Action: REVIEW_CTR


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ga4_sessions,gsc_avg_position,scroll_events,ctr,is_candidate,volume_score,position_score,score,reason_code,action
0,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0,0.181500,0,0.000000,True,1.000000,0.990925,99.637001,LOW_CTR_VISIBLE,REVIEW_CTR
1,2026-03-30,client_73cda7b4e4f265ea,content_8e1334d6356668e3,16076,0,0,0.323090,0,0.000000,True,0.929848,0.983845,95.144725,LOW_CTR_VISIBLE,REVIEW_CTR
2,2026-03-30,client_e547b89c05043229,content_0e03de7680314cd5,19584,10,11,2.401859,4,0.051062,True,0.948798,0.879907,92.124155,LOW_CTR_VISIBLE,REVIEW_CTR
3,2026-03-30,client_23a62021009f63c4,content_fa4cf3aa5ce67bb8,8481,0,0,1.704044,0,0.000000,True,0.868457,0.914798,88.699327,LOW_CTR_VISIBLE,REVIEW_CTR
4,2026-03-30,client_e547b89c05043229,content_8d7d99f109e19aa2,8818,1,1,2.330687,1,0.011340,True,0.872198,0.883466,87.670482,LOW_CTR_VISIBLE,REVIEW_CTR
5,2026-03-30,client_e547b89c05043229,content_4ffe18112a5642e3,8863,21,9,2.365903,0,0.236940,True,0.872686,0.881705,87.629369,LOW_CTR_VISIBLE,REVIEW_CTR
6,2026-03-30,client_e547b89c05043229,content_545bb6cc7081ded3,8458,2,1,2.544810,0,0.023646,True,0.868196,0.872760,87.002155,LOW_CTR_VISIBLE,REVIEW_CTR
7,2026-03-30,client_62f4a7e64f5e0096,content_7172a7fad43f0998,10200,46,<NA>,3.172549,<NA>,0.450980,True,0.886174,0.841373,86.825351,LOW_CTR_VISIBLE,REVIEW_CTR
8,2026-03-30,client_e547b89c05043229,content_dc91779c3d085398,7334,0,0,2.284838,0,0.000000,True,0.854508,0.885758,86.700810,LOW_CTR_VISIBLE,REVIEW_CTR
9,2026-03-30,client_e547b89c05043229,content_9ef3d7516483e665,6726,8,2,2.324710,2,0.118941,True,0.846201,0.883764,86.122623,LOW_CTR_VISIBLE,REVIEW_CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# Build the ranked action queue

import os

queue = baseline[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "score",
        "reason_code",
        "action"
    ]
].copy()

queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Create the output directory if it does not already exist.
os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV.
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Ranked queue written to:", output_path)
print("Rows written:", len(queue))

print("\nTop 10:")
display(queue.head(10))

Ranked queue written to: work/outputs/baseline_action_score.csv
Rows written: 38250

Top 10:


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,1,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,0.000000,99.637001,LOW_CTR_VISIBLE,REVIEW_CTR
1,2,2026-03-30,client_73cda7b4e4f265ea,content_8e1334d6356668e3,16076,0,0.323090,0.000000,95.144725,LOW_CTR_VISIBLE,REVIEW_CTR
2,3,2026-03-30,client_e547b89c05043229,content_0e03de7680314cd5,19584,10,2.401859,0.051062,92.124155,LOW_CTR_VISIBLE,REVIEW_CTR
3,4,2026-03-30,client_23a62021009f63c4,content_fa4cf3aa5ce67bb8,8481,0,1.704044,0.000000,88.699327,LOW_CTR_VISIBLE,REVIEW_CTR
4,5,2026-03-30,client_e547b89c05043229,content_8d7d99f109e19aa2,8818,1,2.330687,0.011340,87.670482,LOW_CTR_VISIBLE,REVIEW_CTR
5,6,2026-03-30,client_e547b89c05043229,content_4ffe18112a5642e3,8863,21,2.365903,0.236940,87.629369,LOW_CTR_VISIBLE,REVIEW_CTR
6,7,2026-03-30,client_e547b89c05043229,content_545bb6cc7081ded3,8458,2,2.544810,0.023646,87.002155,LOW_CTR_VISIBLE,REVIEW_CTR
7,8,2026-03-30,client_62f4a7e64f5e0096,content_7172a7fad43f0998,10200,46,3.172549,0.450980,86.825351,LOW_CTR_VISIBLE,REVIEW_CTR
8,9,2026-03-30,client_e547b89c05043229,content_dc91779c3d085398,7334,0,2.284838,0.000000,86.700810,LOW_CTR_VISIBLE,REVIEW_CTR
9,10,2026-03-30,client_e547b89c05043229,content_9ef3d7516483e665,6726,8,2.324710,0.118941,86.122623,LOW_CTR_VISIBLE,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# Top-20 review
# For each high-priority candidate, record:
#   - action
#   - why it was selected
#   - what could make the recommendation wrong

top20 = queue.head(20).copy()

top20["why_here"] = top20.apply(
    lambda row: (
        f"CTR is {row['ctr']:.3f}% with "
        f"{int(row['gsc_impressions']):,} impressions and "
        f"average position {row['gsc_avg_position']:.2f}; "
        f"this meets the LOW_CTR_VISIBLE rule."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The low CTR may reflect query mix, measurement limitations, "
    "or other context not captured by this baseline."
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "why_here",
        "what_would_make_it_wrong"
    ]
].copy()

display(review)

,rank,client_hash_id,content_hash_id,action,why_here,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,REVIEW_CTR,"CTR is 0.000% with 33,383 impressions and aver...","The low CTR may reflect query mix, measurement..."
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,REVIEW_CTR,"CTR is 0.000% with 16,076 impressions and aver...","The low CTR may reflect query mix, measurement..."
2,3,client_e547b89c05043229,content_0e03de7680314cd5,REVIEW_CTR,"CTR is 0.051% with 19,584 impressions and aver...","The low CTR may reflect query mix, measurement..."
3,4,client_23a62021009f63c4,content_fa4cf3aa5ce67bb8,REVIEW_CTR,"CTR is 0.000% with 8,481 impressions and avera...","The low CTR may reflect query mix, measurement..."
4,5,client_e547b89c05043229,content_8d7d99f109e19aa2,REVIEW_CTR,"CTR is 0.011% with 8,818 impressions and avera...","The low CTR may reflect query mix, measurement..."
5,6,client_e547b89c05043229,content_4ffe18112a5642e3,REVIEW_CTR,"CTR is 0.237% with 8,863 impressions and avera...","The low CTR may reflect query mix, measurement..."
6,7,client_e547b89c05043229,content_545bb6cc7081ded3,REVIEW_CTR,"CTR is 0.024% with 8,458 impressions and avera...","The low CTR may reflect query mix, measurement..."
7,8,client_62f4a7e64f5e0096,content_7172a7fad43f0998,REVIEW_CTR,"CTR is 0.451% with 10,200 impressions and aver...","The low CTR may reflect query mix, measurement..."
8,9,client_e547b89c05043229,content_dc91779c3d085398,REVIEW_CTR,"CTR is 0.000% with 7,334 impressions and avera...","The low CTR may reflect query mix, measurement..."
9,10,client_e547b89c05043229,content_9ef3d7516483e665,REVIEW_CTR,"CTR is 0.119% with 6,726 impressions and avera...","The low CTR may reflect query mix, measurement..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Weak picks + leakage check

# Show lower-ranked candidates so we can inspect weaker picks.
weak_picks = queue.tail(10).copy()

print("=== Weakest 10 candidates in the ranked queue ===")
display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

# Leakage check:
# These are fields that must not be used as ordinary model features.
banned_feature_terms = [
    "label",
    "future",
    "declining",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type"
]

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

leakage_hits = [
    col for col in feature_columns
    if any(term in col.lower() for term in banned_feature_terms)
]

print("\n=== Leakage check ===")
print("Feature columns used by the baseline:")
print(feature_columns)

print("\nBanned-term matches:")
print(leakage_hits)

assert len(leakage_hits) == 0, "Potential leakage found in feature columns."

print("\nLeakage check: PASSED")
print("Decision date used:", df["report_date"].min())
print("No future-window or label-derived fields were used.")

=== Weakest 10 candidates in the ranked queue ===


,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
38240,38241,client_23a62021009f63c4,content_f5ebdebd9dfe17de,22,0,19.909091,0.0,18.243712,LOW_CTR_VISIBLE,REVIEW_CTR
38241,38242,client_fef1a8f436438636,content_b1196fc1a7af4ae9,22,0,19.909091,0.0,18.243712,LOW_CTR_VISIBLE,REVIEW_CTR
38242,38243,client_62f4a7e64f5e0096,content_2bb8319859a45d5e,22,0,19.909091,0.0,18.243712,LOW_CTR_VISIBLE,REVIEW_CTR
38243,38244,client_e5c2aa26a8598242,content_d26ea31a0070185b,20,0,19.700000,0.0,18.137854,LOW_CTR_VISIBLE,REVIEW_CTR
38244,38245,client_23a62021009f63c4,content_cd70c8b6d04d8312,20,0,19.700000,0.0,18.137854,LOW_CTR_VISIBLE,REVIEW_CTR
38245,38246,client_20259bd6705d81d4,content_43ab19b42f9379c5,21,0,19.857143,0.0,18.091546,LOW_CTR_VISIBLE,REVIEW_CTR
38246,38247,client_20259bd6705d81d4,content_68b6fdf7bf59db84,20,0,19.750000,0.0,18.037854,LOW_CTR_VISIBLE,REVIEW_CTR
38247,38248,client_23a62021009f63c4,content_9e548068108b5f7c,20,0,19.800000,0.0,17.937854,LOW_CTR_VISIBLE,REVIEW_CTR
38248,38249,client_e547b89c05043229,content_84e0293183987c37,20,0,19.950000,0.0,17.637854,LOW_CTR_VISIBLE,REVIEW_CTR
38249,38250,client_fef1a8f436438636,content_508f35cfa21950b9,20,0,20.000000,0.0,17.537854,LOW_CTR_VISIBLE,REVIEW_CTR



=== Leakage check ===
Feature columns used by the baseline:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Banned-term matches:
[]

Leakage check: PASSED
Decision date used: 2026-03-30 00:00:00
No future-window or label-derived fields were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.